In [1]:
# ====================== 环境准备 ======================
# 1) 设定环境变量（必须在导入 matplotlib 之前执行）
import os




os.environ['QT_API'] = 'pyqt5'        # 指定使用 PyQt5 作为 Qt 绑定
os.environ['MPLBACKEND'] = 'QtAgg'    # 指定 Matplotlib 后端为 QtAgg（更推荐，替代 TkAgg）

# 2) 启动 Qt 事件循环（控制台模式下，让 Qt 窗口能实时响应）
%gui qt5

# 3) 检查 matplotlib backend
import matplotlib as mpl
mpl.rcParams.update({
    # 这里按系统常见字体给一串候选，存在则自动生效
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "SimSun",
                        "Noto Sans CJK SC", "Source Han Sans SC",
                        "Arial Unicode MS", "DejaVu Sans"],
    "font.family": "sans-serif",
    "axes.unicode_minus": False,   # 负号用正常字符，避免被当作缺字形
})

print("backend (before pyplot):", mpl.get_backend())
# 如果不是 QtAgg，强制改为 QtAgg（注意：必须在导入 pyplot 前设置）
mpl.rcParams['backend'] = 'QtAgg'

# 4) 现在再导入 pyplot
import matplotlib.pyplot as plt
print("backend (after pyplot):", mpl.get_backend())
from config import DATA_DIR,INPUT_DIR
from typing import Dict, Any


# def slice_group_data(raw_group_data, start, end):
#     """
#     从 raw_group_data 中裁剪时间区间 [basicSa, end)
#     """
#     return {
#         step: raw_group_data[step]
#         for step in range(start, end)
#         if step in raw_group_data
#     }

from src.io.operate_group_data import slice_group_data

import src.io.read_snap_xml  as read_snap_xml
from src.config.viewer_config import G60_CONFIG
DATA_DIR = r"C:\usrspace\mywork\data_paper2"


backend (before pyplot): QtAgg
backend (after pyplot): qtagg


In [2]:
from src.viz.pyqt_main2 import SatelliteViewer


from PyQt5 import QtWidgets

In [16]:
import importlib
import src.viz.pyqt_main2 as pyqt_main2

importlib.reload(pyqt_main2)

SatelliteViewer = pyqt_main2.SatelliteViewer

In [4]:
import importlib
import src.viz.group_data_transform as group_data_transform

importlib.reload(group_data_transform)


<module 'src.viz.group_data_transform' from 'D:\\paper3\\generic\\src\\viz\\group_data_transform.py'>

In [5]:
import importlib
import src.config.constellation_config as constellation_config

importlib.reload(constellation_config)


<module 'src.config.constellation_config' from 'D:\\paper3\\generic\\src\\config\\constellation_config.py'>

In [ ]:
# from pathlib import Path
# # 默认用 "topology_{TIME_2_BUILD}"，也允许用环境变量 TOPOLOGY_VERSION 覆盖
# VERSION = os.getenv("TOPOLOGY_VERSION", f"baseline")
#
# RAW_DIR    = Path(INPUT_DIR) / VERSION / "raw"
# CONFIG_DIR = Path(INPUT_DIR) /VERSION / "baselie"


In [ ]:
# def ensure_dirs(*paths: Path):
#     for p in paths:
#         try:
#             p.mkdir(parents=True, exist_ok=True)
#         except FileExistsError:
#             # 目录名已被一个同名“文件”占用
#             if not p.is_dir():
#                 raise NotADirectoryError(f"存在同名文件，无法创建目录: {p}")
#         except Exception as e:
#             raise RuntimeError(f"创建目录失败 {p}: {e}")
#
# # 保证目录存在（多进程下也安全、可重复调用）
# ensure_dirs(RAW_DIR, CONFIG_DIR)
# # ====================== 导入依赖 ======================
# import sys
# # 避免反复执行时 Qt 类重复导入导致崩溃：如果已加载，先删除再导入
# if 'draw.pyqt_draw.pyqt_main2' in sys.modules:
#     del sys.modules['draw.pyqt_draw.pyqt_main2']
#
# from PyQt5 import QtWidgets
# import pyqtgraph as pg
# from draw.pyqt_draw.pyqt_main2 import SatelliteViewer
# import draw.read_snap_xml  as read_snap_xml
# import draw.read_snap_xml  as read_snap_xml
# # 配置 pyqtgraph：开启抗锯齿，关闭 OpenGL（更稳定）
# pg.setConfigOptions(antialias=True)

In [6]:
from pathlib import Path

DATA_DIR = Path(r"D:\paper3")
BASEDIR =  DATA_DIR / "data"

VERSION1 = 'satellitesposition'


xml_file = BASEDIR / VERSION1 / "station_visible_satellites_20250106.xml"
# xml_file = DATA_DIR / "visibile_data" / "G60" / "g60.xml"

In [ ]:
# from pathlib import Path
#
# DATA_DIR = Path(r"C:\usrspace\mywork\data_paper2")
# BASEDIR =  DATA_DIR / "visibile_data"
#
# VERSION1="test"
#
# xml_file = BASEDIR / VERSION1 / "visibility_snapshots.xml"


In [ ]:
# from pathlib import Path
#
# DATA_DIR = Path(r"C:\usrspace\mywork\data_paper2")
# BASEDIR =  DATA_DIR / "visibile_data"
#
# VERSION1="DATA"
#
# xml_file = BASEDIR / VERSION1 / "GW.xml"


In [ ]:
%%sql


In [7]:
# ====================== 基础参数 ======================
# 星座参数：每轨道卫星数 N，轨道平面数 P
N = 36
P = 18

# ====================== 读取数据 ======================

# start_ts = 10717
# # end_ts   = 86399
# end_ts   = 11640
# # 解析 XML 得到 group_data，结构：{time_step: {'groups': {...}}}
# group_data = read_snap_xml.parse_xml_group_data(xml_file, start_ts, end_ts)
# 只做一次：解析大区间
RAW_START, RAW_END = 0, 86164
# 注意，这里是一个恒星日
raw_group_data = read_snap_xml.parse_xml_group_data(xml_file, G60_CONFIG,RAW_START, RAW_END)

#下面是图变换的。
#

In [55]:
# 这里再进行小区间分开，实际上也是进行快速迭代
# 用法（左闭右开
start_ts =0

end_ts =100

group_data = slice_group_data(raw_group_data, start_ts, end_ts)


## 城市对读取
下面的代码，实际上就是读取特定城市的代码

In [64]:
series = read_snap_xml.parse_station_timeseries(xml_file, [5, 7, 9,18,15,19], RAW_START, RAW_END)

In [ ]:
raw_group_data

In [66]:
S6 =  series[0]
S8 =  series[1]
S10 =  series[2]
S19 =  series[3]
S16 =  series[4]
S20 =  series[5]

In [10]:
base_groupid_now = 0

In [11]:
from src.config.constellation_config import ConstellationConfig
constellation_config = ConstellationConfig(
    name="G60",
    N=36,
    P=18,

)
rev_group_data,offset = group_data_transform.modify_group_data(group_data,constellation_config, base_groupid=base_groupid_now)

In [31]:
import importlib
import src.config.viewer_config as vc
importlib.reload(vc)
from src.config.viewer_config import G60_CONFIG

下面主要是为了测试检测我们的图

In [12]:
from src.config.viewer_config import G60_CONFIG
# ====================== 绘图初始化 ======================
# 1) QApplication 实例（全局唯一）
app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])

# 2) 确保 viewer 有全局引用，避免 GC 回收导致崩溃
if not hasattr(sys.modules[__name__], "_viewer_list"):
    _viewer_list = []

In [13]:


# 3) 创建并配置 viewer
viewer = SatelliteViewer(group_data, G60_CONFIG)
viewer.setWindowTitle("group_data")
viewer.resize(1200, 700)
# viewer.edges_by_step =
# viewer.pending_links_by_step =
viewer.show()



In [26]:
# 3) 创建并配置 viewer
viewer = SatelliteViewer(rev_group_data,config=G60_CONFIG)
viewer.setWindowTitle("rev_group_data")
viewer.resize(1200, 700)
# viewer.edges_by_step =
# viewer.pending_links_by_step =
viewer.show()


In [ ]:
# 4) 保存全局引用,只要放入到这个容器里，就能持久存在
_viewer_list.append(viewer)

## 绘制motif
我们的motif代码，实际上除了同构图绘制外，在原图上也可以设计的

In [45]:
# demo 1: 纯 motif 边生成
# 含义：
# - P: 轨道面数
# - N: 每个轨道面的卫星数
# - motif: 在 (plane, sat_in_plane) 网格上重复铺开的局部连边模式

from draw.basic_functio.topology_config import TopologyRecorder

P, N = 18, 36
rec = TopologyRecorder(P, N)

# 这里的 nodes 只是为了兼容你现有 API，真正生成边时会用 rec 里录下来的 motif
nodes = {}

# 1) baseline: +Grid 风格
# option=0 在你当前代码里表示:
# (x, y) -> (x+1, y)
# 也就是“相邻轨道面，同一行”的重复连边
# rec.write_distinct_motif(
#     p_start=0, p_end=17,
#     y_start=0, y_end=35,
#     nodes=nodes,
#     option=1,
# )
# for y in range(0, N):
#     rec.write_distinct_motif(
#         p_start=0, p_end=17,
#         y_start=y, y_end=y+1,
#         nodes=nodes,
#         option=4 if y % 2 == 0 else 1,   # 偶斜上 / 奇斜下
#     )
# rec.write_distinct_motif(
#     p_start=0, p_end=1,
#     y_start=0, y_end=1,
#     nodes=nodes,
#     option=1,
# )

for i in range(P):
    for j in range(N):
        if j% 2 == 0:
            if j==0:
                         rec.write_distinct_motif(
                p_start=i, p_end=i+1,
                y_start=j, y_end=j+1,
               nodes=nodes,
                             option=4,
            )
            else:
                    rec.write_distinct_motif(
                p_start=i, p_end=i+1,
                y_start=j, y_end=j+1,
               nodes=nodes,
                             option=4,
            )
        else:
            if j==N-1:

                  rec.write_distinct_motif(
                p_start=i, p_end=i+1,
                y_start=j-1, y_end=j,
               nodes=nodes,
                           option=1,
            )
            else:
                  rec.write_distinct_motif(
                p_start=i, p_end=i+1,
                y_start=j-1, y_end=j,
               nodes=nodes,
                           option=1,
            )





# for y in range(0, N, 2):
#     rec.write_distinct_motif(
#         p_start=0, p_end=P-1,
#         y_start=y, y_end=y,
#         nodes=nodes,
#         option=4,   # even row: y -> y+1
#     )
#
# for y in range(1, N, 2):
#     rec.write_distinct_motif(
#         p_start=0, p_end=P-1,
#         y_start=y, y_end=y,
#         nodes=nodes,
#         option=1,   # odd row: y -> y-1
#     )


# # 4) 如果你想看一段时间内每个 step 的 motif 边
all_adj = rec.render_adj_range(
    t_start_incl=0,
    t_end_excl=10,
    eval_env={"start_ts": 0, "end_ts": 10},
)
#
# print("\nstep=0 的边数:", sum(len(v) for v in all_adj[0].values()))
# print("step=1 的边数:", sum(len(v) for v in all_adj[1].values()))


In [54]:
# ============================================================
# Grid ×  (交叉网格) motif 配置
# ------------------------------------------------------------
# 对每对相邻行 (y, y+1)（y 偶数）：
#   偶数行 (i, y)   --option=4--> (i+1, y+1)    斜上右
#   奇数行 (i, y+1) --option=1--> (i+1, y)      斜下右
# 两条边在相邻轨道间交叉形成 ×
#
# ⭐ 关键：y_start/y_end 必须同时包含源行和目标行，
#         否则 muban_define 会把目标点裁掉。
# ============================================================

assert N % 2 == 0, "Grid × 要求 N 是偶数，才能两两配对"

for y in range(0, N, 2):               # y = 0, 2, 4, ..., N-2
    # 上行斜连: (i, y) → (i+1, y+1)   —— 用 option=4
    rec.write_distinct_motif(
        p_start=0, p_end=P-1,          # 全部轨道 (P=18 → 0..17)
        y_start=y, y_end=y+1,          # ⭐ 必须包含 y+1
        nodes=nodes,
        option=4,
    )
    # 下行斜连: (i, y+1) → (i+1, y)   —— 用 option=1
    rec.write_distinct_motif(
        p_start=0, p_end=P-1,
        y_start=y, y_end=y+1,          # ⭐ 同样包含 y+1
        nodes=nodes,
        option=1,
    )
all_adj = rec.render_adj_range(
    t_start_incl=0,
    t_end_excl=end_ts,
    eval_env={"start_ts": 0, "end_ts": 10},
)

KeyboardInterrupt: 

NameError: name 'make_edges_bidirectional' is not defined

In [56]:
end_ts

100

In [52]:
viewer = SatelliteViewer( group_data,G60_CONFIG)
viewer.setWindowTitle("rev_group_data with rev_group_data")
viewer.resize(1200, 700)
viewer.edges_by_step = all_adj

viewer.show()
_viewer_list.append(viewer)
# viewer.show_envelopes_static(
#     rects_by_group=rects,
#     expand=0.35,
#     colors=colors,
#     persist=True
# )

In [ ]:

# 注意上述我们是在同构拓扑序列上进行的，因此，我们还要将同构拓扑序列进行还原，同时，我们还要考虑到建链时间约束
# import draw.basic_functio.revdata2rawdata as revdata2rawdata
# # attention ,here  it just composed of the inter-link, intra_link hasn't benn conclued
# raw_inter_edge = revdata2rawdata.revedge2rawedge(all_rev_inter_edge,offset)

In [ ]:


#xiamianshi meiyouyiyi de
viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("groupdata with rawedge")

viewer.resize(1200, 700)
viewer.edges_by_step =raw_inter_edge

viewer.show()


In [ ]:
# 4) 保存全局引用,只要放入到这个容器里，就能持久存在
_viewer_list.append(viewer)

我们要处理好建链时间冲突，因此，下面就是处理冲突的代码

注意，这里其实就是IG 内部要处理的建联图

In [ ]:

# 下面是把边转为node存储，因为这种方式存储会比较方便

##
# import draw.basic_functio.inter_edge2nodes as inter_edge2nodes
# all_nodes = inter_edge2nodes.trans_edge2node(raw_inter_edge,P,N)

## motif 配置
这一章节，我们就要用多种拓扑motif了，注意了，motif的配置，无需在变换的拓扑上操作，直接在原来的图上操作即可，

In [ ]:
## motif设计
# motif设计的原则是,

In [58]:
def make_edges_bidirectional(edge_dict):
    """{basicSa: set(dsts)} -> 双向"""
    new_edges = {}
    for src, dsts in edge_dict.items():
        for dst in dsts:
            new_edges.setdefault(src, set()).add(dst)
            new_edges.setdefault(dst, set()).add(src)
    return new_edges


In [59]:
# 1) 静态 inter 拓扑：只构一次（不要 render_adj_range）
inter_once = rec.render_adj_at(
    t=start_ts,
    eval_env={"start_ts": start_ts, "end_ts": end_ts}
)

# 2) 如果你当前走的是“rev -> raw”流程，用你现有函数转一次
# raw_inter_once = revdata2rawdata.revedge2rawedge({start_ts: inter_once}, offset, N)[start_ts]

# 如果你已经是 raw 坐标设计，直接用 inter_once
raw_inter_once = inter_once

# 3) 你现有的双向化函数，复用
raw_inter_once = make_edges_bidirectional(raw_inter_once)

# 4) 静态总图 = intra ring + static inter
base_neighbors = {
    i * N + j: (i * N + ((j + 1) % N), i * N + ((j - 1) % N))
    for i in range(P) for j in range(N)
}
static_edges = {node: {r, l} for node, (r, l) in base_neighbors.items()}
for src, dsts in raw_inter_once.items():
    static_edges.setdefault(src, set()).update(dsts)

# 5) 按现有 API 组装 step->adj（每个 step 共用同一张图）
all_edges = {step: static_edges for step in range(start_ts, end_ts)}

In [63]:

# 4) 画图验证
viewer = SatelliteViewer(group_data, G60_CONFIG)
viewer.setWindowTitle("verify all_edges_view")
viewer.resize(1400, 800)
viewer.edges_by_step = all_edges
viewer.show()
_viewer_list.append(viewer)



In [ ]:
series = read_snap_xml.parse_station_timeseries(
    xml_file, [5, 7, 9, 18, 15, 19], RAW_START, RAW_END
)

S6, S8, S10, S19, S16, S20 = series

In [67]:
from src.io import read_snap_xml
from src.model.plot_2city_shortest_path import compute_stationpair_min_hops_over_time

# 1) 读地面站可见卫星时序


WIN_START = RAW_START
WIN_END = RAW_START + 99   # 100秒窗口（包含 WIN_START 和 WIN_END）





# 2) 计算 S6-S8 在所有时刻的“最短路径跳数（取所有接入对中的最小值）”
df_s6_s8 = compute_stationpair_min_hops_over_time(
    all_edges=all_adj,      # 你的拓扑: {step: {u: neighbors}}
    paris=S6,               # 站点A
    chongqin=S8,            # 站点B
        steps=(WIN_START, WIN_END),   # 只算这100秒
    # steps=None,             # 用 all_adj 的全部 step（最稳）
    undirected=True,
    return_pair=True,       # 返回 best_s/best_d
    return_path=True,       # 返回 path
    static_topology=True,   # 你是静态拓扑，建议显式开
    precompute_all_pairs=True
)

# 3) 直接看结果（不导出文件）
print(df_s6_s8.head(20))

valid = df_s6_s8[df_s6_s8["min_shortest_path"].notna()]
if not valid.empty:
    best_row = valid.loc[valid["min_shortest_path"].idxmin()]
    print("全时段最优一条：")
    print(best_row[["time", "min_shortest_path", "best_s", "best_d", "path"]])


    time  min_shortest_path best_s best_d path
0      0                0.0    192    192  192
1      1                0.0    192    192  192
2      2                0.0    192    192  192
3      3                0.0    192    192  192
4      4                0.0    192    192  192
5      5                0.0    156    156  156
6      6                0.0    156    156  156
7      7                0.0    156    156  156
8      8                0.0    156    156  156
9      9                0.0    156    156  156
10    10                0.0    156    156  156
11    11                0.0    156    156  156
12    12                0.0    156    156  156
13    13                0.0    156    156  156
14    14                NaN                   
15    15                NaN                   
16    16                NaN                   
17    17                NaN                   
18    18                NaN                   
19    19                NaN                   
全时段最优一条：
time

In [ ]:
# 获取station pairs，
# 这个一般只要运行一次就可以了
series = read_snap_xml.parse_station_timeseries(
    xml_file, [5, 7, 9, 18, 15, 19], WIN_START, WIN_END
)


In [69]:
S6 = series[0]
S8 = series[3]


In [71]:
from src.io import read_snap_xml
from src.model.plot_2city_shortest_path import compute_stationpair_min_hops_over_time
from src.viz.pyqt_main2 import SatelliteViewer
from src.config.viewer_config import ViewerConfig, G60_CONFIG

# 你已有变量：xml_file, RAW_START, all_edges, _viewer_list

# =========================
# 1) 只取 100 秒窗口
# =========================
WIN_START = RAW_START
WIN_END = RAW_START + 99   # 含端点，共100个step（若1step=1s）

# 只截取窗口内拓扑，避免 viewer 和计算时间轴不一致
sub_edges = {t: all_edges.get(t, {}) for t in range(WIN_START, WIN_END + 1)}

# =========================
# 2) 计算 S6-S8 最短跳数（每步取接入对中的最小）
# =========================
df_s6_s8 = compute_stationpair_min_hops_over_time(
    all_edges=sub_edges,
    paris=S6,
    chongqin=S8,
    steps=(WIN_START, WIN_END),
    undirected=True,
    return_pair=True,
    return_path=True,
    static_topology=True,
    precompute_all_pairs=True,
)

print(df_s6_s8[["time", "min_shortest_path", "best_s", "best_d", "path"]].head(20))

# =========================
# 3) 构造仅用于可视化的 group_data（只显示 S6/S8 两组）
# =========================
group_data_s6_s8 = {}
for t in range(WIN_START, WIN_END + 1):
    s6_set = set(S6.get(t, set()))
    s8_set = set(S8.get(t, set()))
    group_data_s6_s8[t] = {
        "groups": {
            0: s6_set,   # 组0 -> S6
            1: s8_set,   # 组1 -> S8
        },
        "all_mentioned": s6_set | s8_set
    }

# 可选：做一个仅S6/S8的viewer配置，让图例更清楚
PAIR_CFG = ViewerConfig(
    name="S6_S8_VERIFY",
    N=G60_CONFIG.N,
    P=G60_CONFIG.P,
    station_groups={
        0: {"name": "S6", "stations": [5]},
        1: {"name": "S8", "stations": [7]},
    },
    group_colors=["#ff4d4f", "#2f54eb"],
)

# =========================
# 4) 把最短路叠加到 viewer（蓝色路径）
# =========================
path_by_step = {}
for row in df_s6_s8.itertuples(index=False):
    p = getattr(row, "path", "")
    if isinstance(p, str) and p.strip():
        path_by_step[int(row.time)] = [int(x) for x in p.split("->")]

viewer = SatelliteViewer(group_data_s6_s8, PAIR_CFG)
viewer.setWindowTitle("verify S6-S8 shortest path (100s)")
viewer.resize(1400, 800)
viewer.edges_by_step = sub_edges
viewer.set_paths(path_by_step)   # 叠加每步最短路（蓝线）

viewer.show()
_viewer_list.append(viewer)


    time  min_shortest_path best_s best_d  \
0      0               12.0    192    324   
1      1               12.0    192    324   
2      2               12.0    192    324   
3      3               12.0    192    324   
4      4               12.0    192    324   
5      5               12.0    192    324   
6      6               12.0    192    324   
7      7               12.0    192    324   
8      8               12.0    192    324   
9      9               12.0    192    324   
10    10               12.0    192    324   
11    11               12.0    192    324   
12    12               12.0    192    324   
13    13               12.0    192    324   
14    14               12.0    192    324   
15    15               12.0    192    324   
16    16               12.0    192    324   
17    17               12.0    192    324   
18    18               12.0    192    324   
19    19               12.0    192    324   

                                                 path 

In [72]:
path_by_step[0]

[192, 191, 154, 153, 188, 187, 222, 221, 256, 255, 290, 289, 324]

In [ ]:
FIGURE_DIR

In [ ]:
# # 假设 all_edges 已经存在且很大
#
# target_steps = range(0, 100) # 0 到 99
#
# # 使用字典推导式进行切片
# sliced_edges = {
#     step: all_edges[step]
#     for step in target_steps
#     if step in all_edges # 确保 key 存在
# }

In [ ]:
FIGURE_DIR = BASEDIR / VERSION1 /"path"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)  # 不存在就创建（包含父目录）

start_ts =0

end_ts =177

In [ ]:
FIGURE_DIR

In [ ]:

from collections import defaultdict
import copy




# ==================== 核心函数 ====================
def sat_id_to_orbit(sat_id,N):
    """卫星 ID → 轨道编号"""
    return sat_id // N

def analyze_station_orbits(group_data, N):
    """
    第一遍扫描：统计每个 station 在整个时间范围内覆盖的所有轨道

    返回:
        station_orbits: {station_id: sorted list of orbit numbers}
    """
    station_orbit_sets = defaultdict(set)

    for time_step, time_data in group_data.items():
        groups = time_data['groups']

        for station_id, satellites in groups.items():
            for sat_id in satellites:
                orbit = sat_id_to_orbit(sat_id, N)
                station_orbit_sets[station_id].add(orbit)

    # 转换为排序列表
    station_orbits = {}
    for station_id, orbit_set in station_orbit_sets.items():
        station_orbits[station_id] = sorted(orbit_set)

    return station_orbits

def filter_group_data_by_orbit(group_data, orbit_selection, N, debug=True):
    """
    过滤 group_data，为每个 station 只保留指定的轨道

    参数:
        group_data: 原始数据字典
        orbit_selection: {station_id: orbit_index} 配置（0-based索引）
        N: 每轨道卫星数
        debug: 是否显示调试信息

    返回:
        new_group_data: 过滤后的数据字典
    """
    # 第一步：扫描所有时间步，建立每个 station 的固定轨道列表
    print("第一步：分析各 station 的轨道覆盖...")
    station_orbits = analyze_station_orbits(group_data, N)

    if debug:
        print("\n【全局轨道分析】")
        for station_id in sorted(station_orbits.keys()):
            orbits = station_orbits[station_id]
            print(f"  Station {station_id}: 覆盖 {len(orbits)} 个轨道 → {orbits}")
        print()

    # 第二步：为每个 station 确定要选择的轨道编号
    station_selected_orbit = {}
    for station_id, orbit_index in orbit_selection.items():
        if station_id not in station_orbits:
            print(f"  警告: Station {station_id} 没有数据，跳过")
            continue

        orbits = station_orbits[station_id]
        if orbit_index < len(orbits):
            selected_orbit = orbits[orbit_index]
            station_selected_orbit[station_id] = selected_orbit

            if debug:
                print(f"  Station {station_id}: 选择索引 {orbit_index} → 轨道 {selected_orbit}")
        else:
            print(f"  警告: Station {station_id} 索引 {orbit_index} 超范围（共 {len(orbits)} 个轨道）")
            station_selected_orbit[station_id] = None

    if debug:
        print()

    # 第三步：过滤每个时间步的数据
    print("第二步：过滤各时间步数据...")
    new_group_data = {}

    for time_step, time_data in group_data.items():
        new_time_data = {
            'groups': {},
            'all_mentioned': set()
        }

        groups = time_data['groups']

        for station_id, satellites in groups.items():
            # 检查是否需要过滤此 station
            if station_id not in orbit_selection:
                # 不在配置中，直接复制
                new_time_data['groups'][station_id] = satellites.copy()
                new_time_data['all_mentioned'].update(satellites)
                continue

            # 获取该 station 应该选择的固定轨道
            selected_orbit = station_selected_orbit.get(station_id)

            if selected_orbit is None:
                # 索引超范围，清空
                new_time_data['groups'][station_id] = set()
                continue

            # 从当前时间步的卫星中，筛选属于选定轨道的卫星
            selected_sats = set()
            for sat_id in satellites:
                orbit = sat_id_to_orbit(sat_id, N)
                if orbit == selected_orbit:
                    selected_sats.add(sat_id)

            new_time_data['groups'][station_id] = selected_sats
            new_time_data['all_mentioned'].update(selected_sats)

        new_group_data[time_step] = new_time_data

    if debug:
        print(f"✓ 过滤完成，共处理 {len(new_group_data)} 个时间步\n")

    return new_group_data



# ==================== 配置区域 ====================
# 为每个 station 指定选择第几个轨道（0-based index）
ORBIT_SELECTION = {
    0: 0,  # Station 0 选第 1 个轨道（0-based，即第二个）
    1: 0,  # Station 1 选第 1 个轨道
}

new_group_data = filter_group_data_by_orbit(group_data, ORBIT_SELECTION, N,debug=True)




In [ ]:
# DUo

def filter_group_data_by_orbit(group_data, orbit_selection, N, debug=True):
    """
    过滤 group_data，为每个 station 只保留指定的轨道（支持单选或多选）。

    参数:
        group_data: 原始数据字典
        orbit_selection: {station_id: orbit_indices} 配置
                         支持格式: {0: 1} 或 {0: [1, 2]} 或 {0: (1, 2)}
        N: 每轨道卫星数
        debug: 是否显示调试信息
    """
    # 辅助函数：将输入标准化为集合，方便后续判断
    def normalize_selection(selection):
        if isinstance(selection, int):
            return {selection}  # 单个整数转为集合
        elif isinstance(selection, (list, tuple, set)):
            return set(selection)  # 列表/元组转为集合
        return set()

    # 第一步：扫描所有时间步，建立每个 station 的固定轨道列表
    print("第一步：分析各 station 的轨道覆盖...")
    station_orbits = analyze_station_orbits(group_data, N)

    if debug:
        print("\n【全局轨道分析】")
        for station_id in sorted(station_orbits.keys()):
            orbits = station_orbits[station_id]
            print(f"  Station {station_id}: 覆盖 {len(orbits)} 个轨道 → {orbits}")
        print()

    # 第二步：为每个 station 确定要选择的实际轨道编号（列表）
    station_selected_orbits_map = {} # 存储 {station_id: {orbit_id, orbit_id...}}

    for station_id, selection_raw in orbit_selection.items():
        if station_id not in station_orbits:
            print(f"  警告: Station {station_id} 没有数据，跳过")
            continue

        existing_orbits = station_orbits[station_id] # 该站实际拥有的轨道列表 [1, 5, 8...]
        wanted_indices = normalize_selection(selection_raw) # 用户想要的索引集合 {0, 1}

        valid_selected_orbits = set()

        for idx in wanted_indices:
            if 0 <= idx < len(existing_orbits):
                real_orbit = existing_orbits[idx]
                valid_selected_orbits.add(real_orbit)
            else:
                print(f"  警告: Station {station_id} 索引 {idx} 超范围（共 {len(existing_orbits)} 个轨道）")

        station_selected_orbits_map[station_id] = valid_selected_orbits

        if debug:
            print(f"  Station {station_id}: 选择索引 {wanted_indices} → 实际轨道 {valid_selected_orbits}")

    if debug:
        print()

    # 第三步：过滤每个时间步的数据
    print("第二步：过滤各时间步数据...")
    new_group_data = {}

    for time_step, time_data in group_data.items():
        new_time_data = {
            'groups': {},
            'all_mentioned': set()
        }

        groups = time_data['groups']

        for station_id, satellites in groups.items():
            # 1. 检查是否在配置名单中
            if station_id not in orbit_selection:
                # 不在配置中，保留原样（或者你可以选择清空，看需求，这里保持原逻辑）
                new_time_data['groups'][station_id] = satellites.copy()
                new_time_data['all_mentioned'].update(satellites)
                continue

            # 2. 获取该 station 允许的轨道集合
            allowed_orbits = station_selected_orbits_map.get(station_id, set())

            if not allowed_orbits:
                # 如果没有选到任何有效轨道，清空该站数据
                new_time_data['groups'][station_id] = set()
                continue

            # 3. 筛选卫星
            selected_sats = set()
            for sat_id in satellites:
                orbit = sat_id_to_orbit(sat_id, N)
                # 核心修改：检查卫星轨道是否在允许的集合中
                if orbit in allowed_orbits:
                    selected_sats.add(sat_id)

            new_time_data['groups'][station_id] = selected_sats
            new_time_data['all_mentioned'].update(selected_sats)

        new_group_data[time_step] = new_time_data

    if debug:
        print(f"✓ 过滤完成，共处理 {len(new_group_data)} 个时间步\n")

    return new_group_data


# 配置示例
ORBIT_SELECTION = {
    0: [0, 1],  # Station 0: 选择第 1 个 和 第 2 个轨道（多选，用列表）
    1: 1,


}

# 调用函数
new_group_data = filter_group_data_by_orbit(group_data, ORBIT_SELECTION, N, debug=True)

In [ ]:
new_group_data

In [ ]:


#xiamianshi meiyouyiyi de
viewer = SatelliteViewer(new_group_data)
viewer.setWindowTitle("groupdata with rawedge")

viewer.resize(1200, 700)
viewer.edges_by_step =all_edges

viewer.show()


In [ ]:
# 某些轨道之间的
# 确保你已经 import 了 plot_intergroup_avg_shortest_path
# import draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path
import  draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path






basename = f"avgspath_g0_4_baseline_{start_ts}_to_{end_ts}"


csv_path = plot_intergroup_avg_shortest_path.export_intergroup_avgspath_to_origin(
    all_edges=all_edges,            # 传入切片后的边
    group_data=new_group_data,      # 传入切片后的组数据
    out_dir=FIGURE_DIR,
    basename=basename,
    group_a=0,
    group_b=1,
    steps=(start_ts, end_ts), # 明确指定当前处理的范围
    undirected=True
)


In [ ]:
import  draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path

for i in range(3):
    for j in range(3):
        # 为每个 station 指定选择第几个轨道（0-based index）
        ORBIT_SELECTION = {
            0: i,  # Station 0 选第 1 个轨道（0-based，即第二个）
            1: j,  # Station 1 选第 1 个轨道
        }

        new_group_data = filter_group_data_by_orbit(group_data, ORBIT_SELECTION, N,debug=True)

        basename = f"avgspath_g0_4_baseline_{i}_to_{j}"


        csv_path = plot_intergroup_avg_shortest_path.export_intergroup_avgspath_to_origin(
            all_edges=all_edges,                # 传入切片后的边
            group_data=new_group_data,      # 传入切片后的组数据
            out_dir=FIGURE_DIR,
            basename=basename,
            group_a=0,
            group_b=1,
            steps=(start_ts, end_ts), # 明确指定当前处理的范围
            undirected=True
        )


In [ ]:
# 某些轨道之间的
# 确保你已经 import 了 plot_intergroup_avg_shortest_path
# import draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path
import  draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path






basename = f"avgspath_g0_4_baseline_{start_ts}_to_{end_ts}"


csv_path = plot_intergroup_avg_shortest_path.export_intergroup_avgspath_to_origin(
    all_edges=all_edges,            # 传入切片后的边
    group_data=group_data,      # 传入切片后的组数据
    out_dir=FIGURE_DIR,
    basename=basename,
    group_a=0,
    group_b=1,
    steps=(start_ts, end_ts), # 明确指定当前处理的范围
    undirected=True
)


In [ ]:
group_data

In [ ]:
import time
# 确保你已经 import 了 plot_intergroup_avg_shortest_path
# import draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path
import  draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path
# ================= 1. 设置测试参数 =================
TOTAL_START = start_ts
TOTAL_END = end_ts     # 测试总长度
CHUNK_SIZE = 10000    # 切片大小
# 你的输出路径


# 确保文件夹存在
if not FIGURE_DIR.exists():
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"=== 开始顺序测试: 范围 {TOTAL_START}-{TOTAL_END}, 分片大小 {CHUNK_SIZE} ===")

# ================= 2. 顺序循环执行 =================
for batch_start in range(TOTAL_START, TOTAL_END, CHUNK_SIZE):

    # 计算当前结束点
    batch_end = min(batch_start + CHUNK_SIZE, TOTAL_END)

    print(f"\n>> 正在处理分片: {batch_start} 到 {batch_end} ...")

    # --- A. 切片 all_edges (提取当前 100s 的拓扑) ---
    target_steps = range(batch_start, batch_end)
    sub_edges = {
        step: all_edges[step]
        for step in target_steps
        if step in all_edges
    }

    # 检查数据是否为空
    if not sub_edges:
        print(f"   [警告] 时间段 {batch_start}-{batch_end} 没有拓扑数据，跳过。")
        continue
    else:
        print(f"   [数据] sub_edges 包含 {len(sub_edges)} 个时刻。")

    # --- B. 切片 group_data (提取当前 100s 的组信息) ---
    # 假设 slice_group_data 是你之前定义好的函数
    sub_group_data = slice_group_data(raw_group_data, batch_start, batch_end)

    # --- C. 执行计算与导出 ---
    basename = f"avgspath_g0_4_baseline_{batch_start}_to_{batch_end}"

    try:
        csv_path = plot_intergroup_avg_shortest_path.export_intergroup_avgspath_to_origin(
            all_edges=sub_edges,            # 传入切片后的边
            group_data=sub_group_data,      # 传入切片后的组数据
            out_dir=FIGURE_DIR,
            basename=basename,
            group_a=0,
            group_b=1,
            steps=(batch_start, batch_end), # 明确指定当前处理的范围
            undirected=True
        )
        print(f"   [成功] 文件已生成: {csv_path}")

    except Exception as e:
        print(f"   [错误] 计算时发生异常: {e}")
        # 如果出错，打印出堆栈以便调试
        import traceback
        traceback.print_exc()

print("\n=== 测试运行结束 ===")

In [ ]:
import time
# 确保你已经 import 了 plot_intergroup_avg_shortest_path
# import draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path
import  draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path
import  draw.pymatlab2.chartalgorithm.plot_2city_shortest_path as plot_2city_shortest_path

# ================= 1. 设置测试参数 =================
TOTAL_START = start_ts
TOTAL_END = end_ts     # 测试总长度
CHUNK_SIZE = 10000    # 切片大小
# 你的输出路径


# 确保文件夹存在
if not FIGURE_DIR.exists():
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"=== 开始顺序测试: 范围 {TOTAL_START}-{TOTAL_END}, 分片大小 {CHUNK_SIZE} ===")

# ================= 2. 顺序循环执行 =================
for batch_start in range(TOTAL_START, TOTAL_END, CHUNK_SIZE):

    # 计算当前结束点
    batch_end = min(batch_start + CHUNK_SIZE, TOTAL_END)

    print(f"\n>> 正在处理分片: {batch_start} 到 {batch_end} ...")

    # --- A. 切片 all_edges (提取当前 100s 的拓扑) ---
    target_steps = range(batch_start, batch_end)
    sub_edges = {
        step: all_edges[step]
        for step in target_steps
        if step in all_edges
    }

    # 检查数据是否为空
    if not sub_edges:
        print(f"   [警告] 时间段 {batch_start}-{batch_end} 没有拓扑数据，跳过。")
        continue
    else:
        print(f"   [数据] sub_edges 包含 {len(sub_edges)} 个时刻。")

    # --- B. 切片 group_data (提取当前 100s 的组信息) ---
    # 假设 slice_group_data 是你之前定义好的函数
    sub_group_data = slice_group_data(raw_group_data, batch_start, batch_end)

    # --- C. 执行计算与导出 ---
    # basename = f"avgspath_g0_4_baseline_{batch_start}_to_{batch_end}"

    try:
        # csv_path = plot_intergroup_avg_shortest_path.export_intergroup_avgspath_to_origin(
        #     all_edges=sub_edges,            # 传入切片后的边
        #     group_data=sub_group_data,      # 传入切片后的组数据
        #     out_dir=FIGURE_DIR,
        #     basename=basename,
        #     group_a=0,
        #     group_b=1,
        #     steps=(batch_start, batch_end), # 明确指定当前处理的范围
        #     undirected=True
        # )

        citys1name = f"city22_{batch_start}_to_{batch_end}"
        csv_path = plot_2city_shortest_path.export_stationpair_min_hops_to_origin(
            sub_edges, S8, S16,
            out_dir=FIGURE_DIR,
            basename=citys1name,
            steps=(batch_start, batch_end),
            undirected=True,
            with_pair=True,
            with_path=False
        )

        # citys2name = f"city2_{batch_start}_to_{batch_end}"
        # csv_path = plot_2city_shortest_path.export_stationpair_min_hops_to_origin(
        #     sub_edges, S8, S16,
        #     out_dir=FIGURE_DIR,
        #     basename=citys2name,
        #     steps=(batch_start, batch_end),
        #     undirected=True,
        #     with_pair=True,
        #     with_path=False
        # )
        # citys3name = f"city3_{batch_start}_to_{batch_end}"
        # csv_path = plot_2city_shortest_path.export_stationpair_min_hops_to_origin(
        #     sub_edges, S10, S20,
        #     out_dir=FIGURE_DIR,
        #     basename=citys3name,
        #     steps=(batch_start, batch_end),
        #     undirected=True,
        #     with_pair=True,
        #     with_path=False
        # )




        print(f"   [成功] 文件已生成: {csv_path}")

    except Exception as e:
        print(f"   [错误] 计算时发生异常: {e}")
        # 如果出错，打印出堆栈以便调试
        import traceback
        traceback.print_exc()

print("\n=== 测试运行结束 ===")